# 🤖 Clase 2: Agentes y Herramientas (Tools)

## Bienvenido a la Semana 2, Clase 2

En esta clase aprenderás:
- ✅ ¿Qué son los agentes de IA?
- ✅ ReAct: Reasoning + Acting
- ✅ Tools: Herramientas para agentes
- ✅ Function calling
- ✅ Crear un agente con búsqueda web (Tavily)
- ✅ Chain of Thought

---

In [1]:
# Instalación
!pip install langchain langchain-anthropic langchain-community tavily-python langsmith -q


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain.tools import tool

load_dotenv()

llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)
print("✅ Configuración completada (usando Claude)")

✅ Configuración completada (usando Claude)


## 🤖 Parte 1: ¿Qué es un Agente?

### Diferencia: Chain vs Agente

**Chain (Cadena)**:
```
Input → Paso 1 → Paso 2 → Paso 3 → Output
(Siempre el mismo camino)
```

**Agente**:
```
Input → Pensar → ¿Qué hacer?
           ↓
      Usar Tool A → Pensar → ¿Listo?
           ↓                    ↓
      Usar Tool B → Pensar → Output
(El agente decide qué hacer)
```

### Características de un Agente

1. **Autonomía**: Decide qué herramientas usar
2. **Razonamiento**: Piensa paso a paso
3. **Iteración**: Puede usar múltiples herramientas
4. **Adaptabilidad**: Se ajusta según los resultados

## 🛠️ Parte 2: Tools (Herramientas)

Las **tools** son funciones que el agente puede usar.

In [3]:
# Tool simple: Calculadora
@tool
def calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática. Input debe ser una expresión matemática válida como '2+2' o '10*5'."""
    try:
        resultado = eval(expresion)
        return f"El resultado es: {resultado}"
    except Exception as e:
        return f"Error: {e}"

# Probar
print(calculadora.invoke("25 * 4 + 10"))

El resultado es: 110


In [ ]:
# Más tools de ejemplo
from datetime import datetime
import random

@tool
def obtener_fecha(dummy: str = "") -> str:
    """Obtiene la fecha y hora actual. No requiere input."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def lanzar_dado(dummy: str = "") -> str:
    """Lanza un dado de 6 caras. No requiere input."""
    resultado = random.randint(1, 6)
    return f"🎲 Resultado del dado: {resultado}"

@tool
def contar_palabras(texto: str) -> str:
    """Cuenta las palabras y caracteres en un texto. Input: texto a analizar."""
    palabras = len(texto.split())
    caracteres = len(texto)
    return f"Palabras: {palabras}, Caracteres: {caracteres}"

# Lista de tools
tools = [calculadora, obtener_fecha, lanzar_dado, contar_palabras]


print(f"✅ {len(tools)} herramientas creadas")

✅ 4 herramientas creadas


## 🧠 Parte 3: Crear un Agente

### Prompt del Agente

In [5]:
# En langchain 1.2+ ya no se necesita crear el prompt manualmente.
# create_agent acepta un system_prompt directamente.
# El agente decide automáticamente qué herramientas usar.

system_prompt = """Eres un asistente útil que puede usar herramientas para responder preguntas.
Usa las herramientas cuando sea necesario. Piensa paso a paso.
Si no necesitas una herramienta, responde directamente."""

print("✅ System prompt definido")

✅ System prompt definido


In [6]:
# Crear agente con la nueva API de langchain 1.2+
# create_agent devuelve un grafo compilado (basado en LangGraph)
agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("✅ Agente creado y listo para usar")

✅ Agente creado y listo para usar


In [7]:
# Probar el agente
# En langchain 1.2+, se invoca con {"messages": [...]}
preguntas = [
    "¿Cuánto es 15 * 8 + 23?",
    "¿Qué fecha es hoy?",
    "Lanza un dado y dime el resultado",
    "Cuenta las palabras en esta frase: La inteligencia artificial es fascinante"
]

for pregunta in preguntas:
    print(f"\n{'='*80}")
    print(f"❓ Pregunta: {pregunta}")
    print("="*80)
    
    resultado = agent_executor.invoke(
        {"messages": [{"role": "user", "content": pregunta}]}
    )
    # La respuesta final es el último mensaje
    respuesta = resultado["messages"][-1].content
    print(f"\n🤖 Respuesta: {respuesta}")


❓ Pregunta: ¿Cuánto es 15 * 8 + 23?



🤖 Respuesta: El resultado de **15 × 8 + 23** es **143**. 🧮

Siguiendo el orden de operaciones (primero la multiplicación y luego la suma):
- 15 × 8 = 120
- 120 + 23 = **143**

❓ Pregunta: ¿Qué fecha es hoy?



🤖 Respuesta: Hoy es **jueves, 5 de marzo de 2026** y la hora actual es las **12:09:43**. ¿En qué más puedo ayudarte?

❓ Pregunta: Lanza un dado y dime el resultado



🤖 Respuesta: ¡El dado ha caído en **4**! 🎲

¿Quieres lanzarlo de nuevo o hay algo más en lo que pueda ayudarte?

❓ Pregunta: Cuenta las palabras en esta frase: La inteligencia artificial es fascinante



🤖 Respuesta: ¡Aquí tienes el análisis de tu frase! 📊

- **Frase:** *"La inteligencia artificial es fascinante"*
- 🔤 **Palabras:** 5
- 🔡 **Caracteres:** 40

Las 5 palabras son: **La** / **inteligencia** / **artificial** / **es** / **fascinante**. ✅


## 🔍 Parte 4: ReAct - Reasoning + Acting

**ReAct** es un patrón donde el agente:

1. **Thought (Pensamiento)**: Razona sobre qué hacer
2. **Action (Acción)**: Ejecuta una herramienta
3. **Observation (Observación)**: Ve el resultado
4. Repite hasta tener la respuesta

### Ejemplo de ReAct

```
Pregunta: "¿Cuánto es 25 * 4 y qué día es hoy?"

Thought: Necesito hacer dos cosas: un cálculo y obtener la fecha
Action: Usar Calculadora con "25 * 4"
Observation: El resultado es 100

Thought: Ahora necesito la fecha
Action: Usar ObtenerFecha
Observation: 2024-02-03

Thought: Tengo toda la información
Answer: 25 * 4 = 100, y hoy es 2024-02-03
```

In [8]:
# Pregunta compleja que requiere múltiples herramientas
pregunta_compleja = """Haz lo siguiente:
1. Calcula 50 * 3
2. Dime qué fecha es hoy
3. Lanza un dado
4. Dame un resumen de todo"""

print("🧠 Ejecutando pregunta compleja...\n")
resultado = agent_executor.invoke(
    {"messages": [{"role": "user", "content": pregunta_compleja}]}
)
respuesta = resultado["messages"][-1].content
print(f"\n✅ Respuesta final:\n{respuesta}")

🧠 Ejecutando pregunta compleja...




✅ Respuesta final:
¡Perfecto! Ya tengo todos los resultados. Aquí va el resumen completo:

---

📊 **Resumen de resultados**

1. **🧮 Cálculo (50 × 3):** El resultado es **150**.

2. **📅 Fecha y hora actual:** Hoy es **5 de marzo de 2026**, y son las **12:09:55**.

3. **🎲 Lanzamiento del dado:** ¡Salió un **6**! ¡El número más alto posible! 🎉

---

¡Todo listo! ¿Hay algo más en lo que pueda ayudarte? 😊


## 🌐 Parte 5: Agente con Búsqueda Web (Tavily)

**Tavily** es una API de búsqueda web optimizada para LLMs.

In [9]:
# Configurar Tavily
from langchain_community.tools.tavily_search import TavilySearchResults

# Crear tool de búsqueda
search_tool = TavilySearchResults(
    max_results=3,
    api_key=os.getenv("TAVILY_API_KEY")
)

# Agregar a las tools
tools_con_busqueda = tools + [search_tool]

print(f"✅ Tool de búsqueda web agregada")
print(f"Total de herramientas: {len(tools_con_busqueda)}")

✅ Tool de búsqueda web agregada
Total de herramientas: 5


C:\Users\Victor\AppData\Local\Temp\ipykernel_34584\1839467413.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(


In [10]:
# Crear agente con búsqueda web
agent_web_executor = create_agent(
    model=llm,
    tools=tools_con_busqueda,
    system_prompt=system_prompt
)

print("✅ Agente con búsqueda web creado")

✅ Agente con búsqueda web creado


In [11]:
# Probar búsqueda web
preguntas_web = [
    "¿Cuál es la última versión de Python?",
    "¿Qué pasó hoy en el mundo de la tecnología?",
    "Busca información sobre GPT-4 Turbo"
]

for pregunta in preguntas_web:
    print(f"\n{'='*80}")
    print(f"🔍 Pregunta: {pregunta}")
    print("="*80)
    
    resultado = agent_web_executor.invoke(
        {"messages": [{"role": "user", "content": pregunta}]}
    )
    respuesta = resultado["messages"][-1].content
    print(f"\n🤖 Respuesta:\n{respuesta}")


🔍 Pregunta: ¿Cuál es la última versión de Python?



🤖 Respuesta:
Aunque no pude acceder a los resultados de búsqueda en este momento, puedo decirte con base en mi conocimiento actualizado que:

🐍 **La última versión estable de Python es la 3.13**, lanzada en octubre de 2024.

Algunos puntos destacados de **Python 3.13**:
- **Mejoras en el intérprete interactivo (REPL)**: más colorido y con mejor manejo de errores.
- **Mensajes de error mejorados**: más claros y descriptivos.
- **Modo de subprocesos sin GIL (experimental)**: avance hacia la eliminación del Global Interpreter Lock.
- **Compilador JIT (experimental)**: para mejorar el rendimiento.
- **Mejoras en el módulo `typing`** y otras bibliotecas estándar.

> ⚠️ Te recomiendo verificar la versión más reciente directamente en el sitio oficial: [python.org](https://www.python.org/downloads/), ya que puede haber actualizaciones menores (como 3.13.x) posteriores a mi conocimiento.

🔍 Pregunta: ¿Qué pasó hoy en el mundo de la tecnología?



🤖 Respuesta:
Hoy es **5 de marzo de 2026**. Sin embargo, tuve un problema al intentar acceder al buscador de noticias en este momento (error de conexión con la fuente de búsqueda), por lo que no puedo traerte las noticias más recientes del día de forma automática.

Aquí tienes algunas alternativas para mantenerte informado:

1. 🌐 **Webs de tecnología recomendadas:**
   - [Xataka](https://www.xataka.com) *(en español)*
   - [TechCrunch](https://techcrunch.com) *(en inglés)*
   - [The Verge](https://www.theverge.com) *(en inglés)*
   - [Genbeta](https://www.genbeta.com) *(en español)*
   - [Hipertextual](https://hipertextual.com) *(en español)*

2. 📱 **Redes sociales:** Sigue hashtags como `#tecnología`, `#tech` o `#IA` en X (Twitter) o LinkedIn.

3. 🔍 **Búsqueda directa:** Puedes buscar en Google: *"noticias tecnología hoy 5 marzo 2026"*.

Disculpa los inconvenientes. ¿Hay algún tema tecnológico específico sobre el que quieras que te hable o ayude? 😊

🔍 Pregunta: Busca información sobr


🤖 Respuesta:
Lo siento, parece que hubo un problema de autenticación con el motor de búsqueda en este momento. Sin embargo, puedo darte información sobre **GPT-4 Turbo** basada en mi conocimiento:

---

## 🤖 GPT-4 Turbo

**GPT-4 Turbo** es un modelo de lenguaje avanzado desarrollado por **OpenAI**, presentado en noviembre de 2023. Es una versión mejorada y más eficiente de GPT-4. Aquí sus características principales:

### 📌 Características Clave

| Característica | Detalle |
|---|---|
| **Ventana de contexto** | Hasta **128,000 tokens** (~300 páginas de texto) |
| **Conocimiento actualizado** | Datos hasta **abril de 2023** |
| **Multimodalidad** | Acepta texto e imágenes como entrada |
| **Precio** | Más económico que GPT-4 estándar |
| **Velocidad** | Más rápido que GPT-4 original |

### 🚀 Mejoras respecto a GPT-4

- **Mayor contexto**: Puede procesar documentos mucho más largos.
- **Mejor seguimiento de instrucciones**: Más preciso al seguir indicaciones del usuario.
- **Conocimien

## 🎯 Parte 6: Tools Personalizadas Avanzadas

In [12]:
# Tool que accede a una "base de datos" ficticia
productos_db = {
    "laptop": {"nombre": "Laptop Pro", "precio": 1200, "stock": 5},
    "mouse": {"nombre": "Mouse Inalámbrico", "precio": 25, "stock": 50},
    "teclado": {"nombre": "Teclado Mecánico", "precio": 80, "stock": 15}
}

@tool
def consultar_producto(producto: str) -> str:
    """Consulta información de un producto (laptop, mouse, teclado). Input: nombre del producto."""
    producto = producto.lower().strip()
    
    if producto in productos_db:
        info = productos_db[producto]
        return f"""Producto: {info['nombre']}
Precio: ${info['precio']}
Stock disponible: {info['stock']} unidades"""
    else:
        return f"Producto '{producto}' no encontrado. Productos disponibles: {', '.join(productos_db.keys())}"

@tool
def calcular_descuento(input_str: str) -> str:
    """Calcula precio con descuento. Input: 'precio,descuento' (ej: '100,20' para $100 con 20% descuento)."""
    try:
        precio_num, desc_num = map(float, input_str.split(','))
        precio_final = precio_num * (1 - desc_num/100)
        ahorro = precio_num - precio_final
        return f"Precio original: ${precio_num}\nDescuento: {desc_num}%\nPrecio final: ${precio_final:.2f}\nAhorro: ${ahorro:.2f}"
    except:
        return "Error: Formato debe ser 'precio,descuento' (ej: '100,20')"

# Lista de tools para la tienda
tools_tienda = [consultar_producto, calcular_descuento, calculadora]

print("✅ Tools de tienda creadas")

✅ Tools de tienda creadas


In [13]:
# Crear agente de tienda
agent_tienda_executor = create_agent(
    model=llm,
    tools=tools_tienda,
    system_prompt="""Eres un asistente de ventas de una tienda de tecnología.
Ayudas a los clientes a consultar productos y calcular precios.
Sé amable y profesional."""
)

print("✅ Agente de tienda creado")

✅ Agente de tienda creado


In [14]:
# Probar agente de tienda
consultas_tienda = [
    "¿Cuánto cuesta la laptop?",
    "Si compro 3 mouse, ¿cuánto pagaría en total?",
    "¿Cuál sería el precio del teclado con 15% de descuento?"
]

for consulta in consultas_tienda:
    print(f"\n{'='*80}")
    print(f"👤 Cliente: {consulta}")
    print("="*80)
    
    resultado = agent_tienda_executor.invoke(
        {"messages": [{"role": "user", "content": consulta}]}
    )
    respuesta = resultado["messages"][-1].content
    print(f"\n🛍️ Asistente: {respuesta}")


👤 Cliente: ¿Cuánto cuesta la laptop?



🛍️ Asistente: ¡Aquí tienes la información! 🖥️

- **Producto:** Laptop Pro
- **Precio:** $1,200
- **Stock disponible:** 5 unidades

Tenemos buena disponibilidad. ¿Te gustaría saber si hay algún descuento disponible o tienes alguna otra pregunta? 😊

👤 Cliente: Si compro 3 mouse, ¿cuánto pagaría en total?



🛍️ Asistente: ¡Aquí tienes el resumen de tu compra! 🛒

| Producto | Precio unitario | Cantidad | Total |
|---|---|---|---|
| Mouse Inalámbrico | $25 | 3 | **$75** |

Por **3 mouse inalámbricos** pagarías un total de **$75**. Además, contamos con **50 unidades en stock**, así que no hay problema con la disponibilidad. 😊

¿Te gustaría saber algo más o agregar otro producto?

👤 Cliente: ¿Cuál sería el precio del teclado con 15% de descuento?



🛍️ Asistente: ¡Aquí tienes el resumen! 🎉

| Detalle | Valor |
|---|---|
| 🖮 Producto | Teclado Mecánico |
| 💵 Precio original | $80.00 |
| 🏷️ Descuento | 15% |
| ✅ **Precio final** | **$68.00** |
| 💰 Ahorro | $12.00 |

Con el **15% de descuento**, el **Teclado Mecánico** te quedaría en solo **$68.00**, ahorrándote **$12.00**. Además, contamos con **15 unidades en stock** disponibles. 😊

¿Te gustaría proceder con la compra o tienes alguna otra consulta?


## 💡 Ejercicios Prácticos

In [15]:
# Ejercicio 1: Crea tu propia tool
# Ejemplo: Una tool que convierta temperaturas

@tool
def convertir_temperatura(input_str: str) -> str:
    """
    Convierte temperatura entre Celsius y Fahrenheit.
    Input: 'valor,unidad' (ej: '25,C' o '77,F')
    """
    try:
        valor, unidad = input_str.split(',')
        valor = float(valor)
        unidad = unidad.strip().upper()
        
        if unidad == 'C':
            fahrenheit = (valor * 9/5) + 32
            return f"{valor}°C = {fahrenheit:.1f}°F"
        elif unidad == 'F':
            celsius = (valor - 32) * 5/9
            return f"{valor}°F = {celsius:.1f}°C"
        else:
            return "Error: Unidad debe ser 'C' o 'F'"
    except:
        return "Error: Formato debe ser 'valor,unidad' (ej: '25,C')"

# Pruébalo
print(convertir_temperatura.invoke("25,C"))
print(convertir_temperatura.invoke("77,F"))

25.0°C = 77.0°F
77.0°F = 25.0°C


In [16]:
# Ejercicio 2: Agente con memoria (conversación)
# En langchain 1.2+ usamos checkpointer para persistir estado entre invocaciones
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

# Agente conversacional con memoria
agent_conversacional = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Eres un asistente amigable. Recuerda lo que el usuario te dice.",
    checkpointer=checkpointer
)

# Conversación de ejemplo (usamos thread_id para mantener el hilo)
config = {"configurable": {"thread_id": "conversacion-1"}}

print("Conversación 1:")
resultado1 = agent_conversacional.invoke(
    {"messages": [{"role": "user", "content": "Mi nombre es Juan"}]},
    config=config
)
print(resultado1["messages"][-1].content)

print("\nConversación 2:")
resultado2 = agent_conversacional.invoke(
    {"messages": [{"role": "user", "content": "¿Cuál es mi nombre?"}]},
    config=config
)
print(resultado2["messages"][-1].content)

Conversación 1:


¡Hola, **Juan**! 😊 Es un placer conocerte. Aquí estoy para ayudarte en lo que necesites. ¿En qué puedo asistirte hoy?

Conversación 2:


¡Tu nombre es **Juan**! 😊 Me lo dijiste hace un momento. ¿Hay algo más en lo que pueda ayudarte?


## 🎓 Resumen

### Conceptos Clave

1. **Agentes**: Sistemas que deciden qué hacer
2. **Tools**: Funciones que los agentes pueden usar
3. **ReAct**: Patrón de razonamiento + acción
4. **Function Calling**: LLM invoca funciones
5. **Tavily**: Búsqueda web para agentes
6. **AgentExecutor**: Ejecuta el agente con límites

### Mejores Prácticas

- ✅ Descripciones claras de tools
- ✅ Manejo de errores en tools
- ✅ Límite de iteraciones
- ✅ Verbose=True para debugging
- ✅ Temperatura baja (0-0.3) para agentes

### Próxima Semana

En **Semana 3** aprenderemos:
- 📊 LangGraph en profundidad
- 🔄 State management
- 🤖 Sistemas multi-agente
- 📈 LangSmith para monitoreo

---

**¡Excelente trabajo! 🚀**